In [ ]:
!pip -q install -U transformers accelerate openpyxl

from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import numpy as np, torch
from pathlib import Path

DRIVE = Path('/content/drive/MyDrive/diss-datasets')
RESULTS = Path('/content/drive/MyDrive/results.xlsx')
OUT = DRIVE / 'siglip2_embeddings'
OUT.mkdir(exist_ok=True)

MODEL_ID = 'google/siglip2-base-patch16-224'
RES = 224
DEVICE = 'cpu'
ORDER = ['CHBMIT']

DATASETS = {
    'CHBMIT': 'CHBMIT_crosspatient_v2.npy',
    'CMJ': 'CMJ.npy',
    'MP8': 'MP8.npy',
    'MP50': 'MP50.npy',
    'SYNTH': 'SYNTH.npy',
    'Rowing': 'Rowing.npy',
}

def _unwrap(o):
    if isinstance(o, np.ndarray) and o.dtype == object and o.ndim == 0:
        return o.item()
    return o

def _xy(split):
    s = _unwrap(split)
    xk = next(k for k in s.keys() if k.lower() in ('x', 'data', 'inputs', 'samples'))
    yk = next(k for k in s.keys() if k.lower() in ('y', 'labels', 'label', 'targets'))
    return np.asarray(s[xk]), np.asarray(s[yk])

def load_dataset(name):
    d = _unwrap(np.load(DRIVE / DATASETS[name], allow_pickle=True))
    Xtr, ytr = _xy(d['train'])
    Xte, yte = _xy(d['test'])
    if Xtr.ndim == 2:
        Xtr, Xte = Xtr[:, None, :], Xte[:, None, :]
    assert Xtr.shape[1] == Xte.shape[1], f'{name}: channel mismatch'
    return (Xtr.astype(np.float32), np.asarray(ytr).ravel(),
            Xte.astype(np.float32), np.asarray(yte).ravel())

def load_meta(name, split):
    s = _unwrap(_unwrap(np.load(DRIVE / DATASETS[name], allow_pickle=True))[split])
    return {k: np.asarray(s[k]) for k in s.keys() if k not in ('X', 'y')}

for name in ORDER:
    Xtr, ytr, Xte, yte = load_dataset(name)
    print(name, Xtr.shape, Xte.shape, 'classes:', len(np.unique(ytr)))

In [ ]:
import torch.nn.functional as F
import matplotlib.pyplot as plt

def robust_scale(x):
    q1, q2, q3 = np.percentile(x, [25, 50, 75], axis=-1, keepdims=True)
    iqr = q3 - q1
    iqr[iqr == 0] = 1.0
    return (x - q2) / iqr

def to_stacked_gray(X, res=RES, chunk=256):
    N, D, T = X.shape
    out = torch.empty(N, 1, res, res, dtype=torch.float32)
    for i in range(0, N, chunk):
        blk = X[i:i + chunk]
        n = len(blk)
        s = robust_scale(blk.reshape(n * D, T)).reshape(n, D, T)
        t = torch.from_numpy(s).float().unsqueeze(1)
        lo = t.amin(dim=(2, 3), keepdim=True)
        hi = t.amax(dim=(2, 3), keepdim=True)
        t = (t - lo) / (hi - lo + 1e-8)
        t = (t - 0.5) * 0.8 + 0.5
        out[i:i + chunk] = F.interpolate(t, size=(res, res), mode='area')
    return out

Xtr, ytr, Xte, yte = load_dataset('CHBMIT')
demo = to_stacked_gray(Xtr[:1])
plt.figure(figsize=(3, 3))
plt.imshow(demo[0, 0].numpy(), cmap='gray')
plt.axis('off')
plt.title('CHBMIT stacked, 18 channels x time')
plt.show()

In [ ]:
from transformers import AutoModel

model = AutoModel.from_pretrained(MODEL_ID, torch_dtype=torch.float32).to(DEVICE).eval()
vision = model.vision_model
N_LAYERS = model.config.vision_config.num_hidden_layers
HID = model.config.vision_config.hidden_size
LAYER = max(1, round(0.6 * N_LAYERS))
print(N_LAYERS, 'layers,', HID, 'dims, using layer', LAYER)

@torch.no_grad()
def embed_gray(gray, layer, batch=16):
    outs = []
    for i in range(0, len(gray), batch):
        chunk = gray[i:i + batch].repeat(1, 3, 1, 1)
        chunk = ((chunk - 0.5) / 0.5).to(DEVICE)
        hs = vision(pixel_values=chunk, output_hidden_states=True).hidden_states
        outs.append(hs[layer].mean(dim=1).float().cpu())
        if i % 320 == 0:
            print(f'  {i}/{len(gray)}', flush=True)
    return torch.cat(outs).numpy().astype(np.float16)

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score

stacked_rows = []
for name in ORDER:
    Xtr, ytr, Xte, yte = load_dataset(name)
    if Xtr.shape[1] < 3:
        continue
    print(name, 'train')
    etr = embed_gray(to_stacked_gray(Xtr), LAYER)
    print(name, 'test')
    ete = embed_gray(to_stacked_gray(Xte), LAYER)
    np.save(OUT / f'SigLIP2stack_{name}_train_emb.npy', etr)
    np.save(OUT / f'SigLIP2stack_{name}_test_emb.npy', ete)

    A, B = etr.astype(np.float32), ete.astype(np.float32)
    lr = LogisticRegression(max_iter=3000, n_jobs=-1).fit(A, ytr)
    p = lr.predict(B)
    stacked_rows.append({
        'dataset': name, 'backbone': 'SigLIP2-base', 'layer': LAYER,
        'depth_frac': round(LAYER / N_LAYERS, 2), 'n_features': int(A.shape[1]),
        'stacked_acc': accuracy_score(yte, p),
        'stacked_macro_f1': f1_score(yte, p, average='macro'),
    })
    if name == 'CHBMIT':
        np.savez(OUT / 'SigLIP2stack_CHBMIT_test_scores.npz',
                 scores=lr.predict_proba(B)[:, 1], y=yte, **load_meta(name, 'test'))
    print(stacked_rows[-1])

In [ ]:
import pandas as pd, shutil
from datetime import datetime

backup = RESULTS.with_name(f'results_backup_{datetime.now():%Y%m%d_%H%M}.xlsx')
shutil.copy(RESULTS, backup)
print('backup:', backup.name)

with pd.ExcelWriter(RESULTS, engine='openpyxl', mode='a',
                    if_sheet_exists='replace') as w:
    pd.DataFrame(stacked_rows).to_excel(w, sheet_name='SigLIP2 stacked', index=False)

print(pd.ExcelFile(RESULTS).sheet_names)
print(pd.DataFrame(stacked_rows))